In [ ]:
import sys, kardemumma
print(sys.executable)         # kernel Python
print(kardemumma.__file__)    # should point to your local repo under src/kardemumma

In [ ]:
# Load packages
from pathlib import Path
import os
import re
import sys
import importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyteomics import achrom
# Importantly, import kardemumma :)
import kardemumma as kdm

In [ ]:

# Skyline data path
skyline_path = "data/DA4000/DA4k_p1-8.csv"

# Import skyline data (includes column "Isotope Label Type" from Precursor)
skyline_importer = kdm.ImportSkylineFile(skyline_path)
skyline_data = skyline_importer.import_skyline_file()

In [ ]:
# **SDRF**
sdrf_path = 'data/DA4000/20260523_DA4K.sdrf.tsv'

# Valdate sdrf 
# kdm.validate_sdrf(sdrf_path)
kdm.readout_ms_type(sdrf_path)
# Import SDRF file for this project

sdrf_data = kdm.ImportSDRFFile(sdrf_path)
sdrf_data_file = sdrf_data.import_sdrf_file()



In [ ]:
# Print out what Peptide Sequence iRT

kdm.get_irt_peptides(skyline_data)

In [ ]:
# Checking possible QC samples from Skyline results

qc_samples = skyline_importer.suggest_qc_samples(skyline_data)
# skyline_checker = CheckSkylineFile(skyline_path)

# Suggest qc samples

print(qc_samples)


In [ ]:
# Plot retention time

skyline_rt = skyline_data.copy()

# Calculate mean retention time for each peptide
mean_rt = skyline_data.groupby('Peptide Sequence')['Predicted Retention Time'].mean()

# Sort peptides by mean retention time
sorted_peptides = mean_rt.sort_values().index.tolist()

# Calculate predicted RT using pyteomics.achrom's calculate_RT with Guo coefficients
skyline_rt['RT'] = skyline_rt['Peptide Sequence'].apply(lambda seq: achrom.calculate_RT(seq, achrom.RCs_guo_ph7_0))

skyline_rt.head()

In [ ]:
# Plot matching between predicted RT and experimental Predicted Retention Time

plt.figure(figsize=(8, 6))
plt.scatter(
    skyline_rt['RT'],
    skyline_rt['Predicted Retention Time'],
    alpha=0.6,
    s=20
)
plt.xlabel('Predicted RT (achrom Guo, pH 7.0)')
plt.ylabel('Experimental RT (Skyline (Koina): Predicted Retention Time)')
plt.title('Predicted vs. Experimental Retention Time for Peptides')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Remove QC samples from skyline_data
skyline_data = kdm.remove_qc_samples(skyline_data, qc_samples)


In [ ]:
kdm.plot_library_dot_product_distribution(skyline_data)

In [ ]:
skyline_clean = kdm.filter_library_dot_product(skyline_data, threshold=0.6)


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = kdm.summarise_peptide_counts(skyline_clean)


In [ ]:
report_summary, peptide_list = kdm.report_peptide_protein_summary(peptide_counts)

In [ ]:
kdm.plot_heavy_light_scatter(peptide_counts)

In [ ]:
filtered_peptide_counts = kdm.filter_peptide_counts(peptide_counts, light_cutoff=600, heavy_cutoff=600)

filtered_peptide_counts.head()


In [ ]:
selected_peptides_report, selected_peptides = kdm.report_peptide_protein_summary(filtered_peptide_counts)

In [ ]:
# from kardemumma.importer import MergeFiles

skyline_merge_obj = kdm.MergeFiles(skyline_data, sdrf_data_file, selected_peptides)
skyline_merge = skyline_merge_obj.merge_files()

In [ ]:
skyline_merge.head(20)

In [ ]:
skyline_pool = skyline_merge_obj.select_pool_data(col_sample='characteristics[Sample]', sample_value='PlasmaPool')
# Need to check printing????

In [ ]:
skyline_pool.head()

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
kdm.plot_pool_boxplot(skyline_pool)


In [ ]:
# Calculate intra-plate CV

peptide_plate_stats = kdm.calculate_intra_plate_cv(skyline_pool, col_name='characteristics[plate]')
kdm.plot_intra_plate_cv_stats(peptide_plate_stats, col_name='characteristics[plate]')

In [ ]:
interplate_cv = kdm.calculate_inter_plate_cv(peptide_plate_stats)
kdm.plot_inter_plate_cv_kde(interplate_cv)

In [ ]:
interplate_cv.head()

In [ ]:
# Print interplate_cv froom low to high
print('This is the interplate_cv from low to high:')
print(interplate_cv[interplate_cv['inter_plate_cv'] < 0.1])

In [ ]:
kdm.plot_cumulative_peptide_count_by_cv(interplate_cv)

In [ ]:
selected_peptides = kdm.get_lowest_cv_peptides(interplate_cv, 10)
selected_peptides

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization

pool_selected_df = skyline_pool[skyline_pool['Peptide Sequence'].isin(selected_peptides)]

# Remove bad plasmapool 

bad_pools = [
       'PRM_20260422_DA_4K_B2_Plate_7_C1.raw',
       'PRM_20260422_DA_4K_B2_Plate_8_B8.raw']


# Remove bad peptides 
bad_peptides = ['ESDTSYVSLK', 'VTLNGVPAQPLGPR', 'FSIEGSYQLEK', 'GLLSGWAR']

# Filter bad_pools from pool_selected_df
pool_selected_df = pool_selected_df[~pool_selected_df['File Name'].isin(bad_pools)]

# Filter bad_peptides from pool_selected_df
pool_selected_df = pool_selected_df[~pool_selected_df['Peptide Sequence'].isin(bad_peptides)]

# Filter pool_data to include only those peptides for normalization calculation
pool_selected_df

In [ ]:
kdm.plot_pool_boxplot(pool_selected_df)

In [ ]:
plate_factor_table, conversion_factors, model = kdm.get_plate_conversion_factors(pool_selected_df, col_plate='characteristics[plate]', log_transform=True  )

In [ ]:
plate_factor_table

In [ ]:
conversion_factors

In [ ]:
kdm.plot_plate_conversion_factors(pool_selected_df, col_plate='characteristics[plate]', log_transform=True  )

In [ ]:
# Use the adjust_ratio_by_plate function above to add RatioLightToHeavy_adj to skyline_merge
# First, construct a Plate column of the correct type for the function
skyline_merge_adj = skyline_merge.copy()
# Remove samples from Replicate column that are in qc_samples
skyline_merge_adj = skyline_merge_adj[~skyline_merge_adj['Replicate'].isin(qc_samples)]

# Remove sampels from Plate_1 from characteristics[plate]
skyline_merge_adj = skyline_merge_adj[skyline_merge_adj['characteristics[plate]'] != 'Plate_1']


In [ ]:
# Adjust ratio by conversion factor 
skyline_merge_adj = kdm.adjust_ratio_by_plate(skyline_merge_adj, conversion_factors)

skyline_merge_adj.head()

In [ ]:
# From skyline_merge_adj, filter to pool samples
pool_data_adj = skyline_merge_adj[skyline_merge_adj['characteristics[Sample]'] != 'PlasmaPool'].copy()

# Reorder by characteristics[Plate]
pool_data_adj = pool_data_adj.sort_values(by='characteristics[plate]')

pool_data_adj.head()


In [ ]:
kdm.plot_pool_boxplot(pool_data_adj)

In [ ]:
pool_selected_adj = pool_data_adj[pool_data_adj['Peptide Sequence'].isin(selected_peptides)]

kdm.plot_plate_conversion_factors(pool_selected_adj, col_plate='characteristics[plate]', log_transform=True  )

# SDRF metadata

Overview of metadata

In [ ]:
import matplotlib.pyplot as plt

# Make a barplot of 'factor value[disease]' column
plt.figure(figsize=(12,6))
value_counts = sdrf_data_file['factor value[disease]'].value_counts()
ax = value_counts.plot(kind='bar')
plt.xlabel('Disease')
plt.ylabel('Count')
plt.title("Sample count by 'factor value[disease]'")
plt.tight_layout()

# Add number on top of each bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2, p.get_height()), 
                ha='center', va='bottom', fontsize=10)

plt.show()

# Calculate absolute quantification

In [ ]:
# kdm.validate_sdrf(sdrf_path)

# Need to fix this 

In [ ]:
qreps_lot = sdrf_data.extract_qreps_lot_number()
print(qreps_lot)

In [ ]:
# Extract qRePS lot from SDRF file 
qreps_lot = sdrf_data.extract_qreps_lot_number()
# Fetch qRePS table
qreps_table = kdm.fetch_qreps_table(qreps_lot)

qreps_table.head()


In [ ]:
# plot_peptide_concentration_by_group is defined in the cell below (after abs_df)


In [ ]:
target_fasta = 'https://proteomedge.com/download/DE17501/DE17501_sequences.fasta'

# Read fasta file
fasta_file = kdm.fetch_fasta(qreps_lot)
fasta_file.head()

In [ ]:
skyline_merge_adj.head()

In [ ]:
# Calculate absolute protein concentration
abs_df = kdm.get_absolute_conc(qreps_table, skyline_merge_adj)

In [ ]:
def map_peptide_sequence(abs_df, fasta_file, protein_name, line_length=30):
    """
    Plot a protein sequence in blocks of N amino acids (multi-line, like in FASTA),
    exactly overlaying each amino acid letter in a "table/box" style, and highlight peptide sequence locations.

    Args:
        abs_df (pd.DataFrame): Peptide concentration data (may be wide or long format).
        fasta_file (pd.DataFrame): Fasta/sequence dataframe (e.g., output of kdm.fetch_fasta()).
        protein_name (str): Protein name (same string as in abs_df['Protein Name'] or MultiIndex level).
        line_length (int): Number of amino acids per row in the plot.

    Returns:
        matplotlib.figure.Figure
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    # ---- Handle wide and long format for abs_df ----
    if isinstance(abs_df.columns, pd.MultiIndex):
        abs_df_flat = abs_df.copy()
        abs_df_flat.columns = [
            '|'.join([str(x) for x in col if x != '' and x is not None])
            for col in abs_df_flat.columns.values
        ]
    else:
        abs_df_flat = abs_df

    def _get_column_if_exists(df, colname):
        if colname in df.columns:
            return df[colname]
        elif df.index.names and colname in df.index.names:
            return df.index.get_level_values(colname)
        else:
            raise KeyError(f"'{colname}' not found in DataFrame columns or index.")

    try:
        protein_name_col = _get_column_if_exists(abs_df_flat, 'Protein Name')
        protein_mask = protein_name_col == protein_name
        if not protein_mask.any():
            raise ValueError(
                f"No rows found in abs_df for Protein Name '{protein_name}'. "
                "Check wide/long format and index levels."
            )
        peptide_seq_col = _get_column_if_exists(abs_df_flat, 'Peptide Sequence')
        peptides = peptide_seq_col[protein_mask].unique()
    except Exception as e:
        raise RuntimeError(f"Could not extract peptide sequences for protein_name='{protein_name}': {e}")

    # --- Find PrEST fasta row for the requested protein ---
    matching = fasta_file[fasta_file['id'] == protein_name]
    if matching.empty:
        stripped = protein_name.split('|')[0]
        matching = fasta_file[fasta_file['id'].str.startswith(stripped)]
        if matching.empty:
            raise ValueError(f"Could not find fasta row for '{protein_name}' (tried '{protein_name}' and '{stripped}')")
    fasta_row = matching.iloc[0]
    sequence = fasta_row['sequence']
    seq_len = len(sequence)
    n_lines = int(np.ceil(seq_len / line_length))

    # Generate peptide position masks for highlighting
    pep_positions = np.zeros(seq_len, dtype=int)  # 0=no, >0, number of overlapping peptides

    pep_indices = {}  # for color mapping and legend
    for idx, pep in enumerate(peptides):
        start = sequence.find(pep)
        if start == -1:
            print(f"Warning: peptide '{pep}' not found in protein sequence for '{protein_name}'.")
            continue
        pep_indices[pep] = idx
        pep_positions[start:start+len(pep)] = idx+1  # store a positive index for highlight association

    # --- Start multi-line "table/box" plotting ---
    fig_height = max(n_lines * 1.3 + 1.5, 2.8)
    fig_width = min(1.3*line_length, 28)
    fig, axes = plt.subplots(n_lines, 1, figsize=(fig_width, fig_height), sharex=False, squeeze=False)

    # Add a big title at the top (protein name, as requested)
    fig.suptitle(str(protein_name), fontsize=18, y=1.0, weight='bold')

    colors = plt.cm.tab20.colors

    for line in range(n_lines):
        ax = axes[line, 0]
        start_idx = line * line_length
        end_idx = min((line + 1) * line_length, seq_len)
        n_aa_in_line = end_idx - start_idx

        # For the last line, to preserve box layout as previous lines, pad with empty boxes if needed
        if n_aa_in_line < line_length:
            pad_end = line_length - n_aa_in_line
        else:
            pad_end = 0

        seq_sub = sequence[start_idx:end_idx]
        x_pos = np.arange(line_length)

        # Draw boxes for each amino acid
        for i in range(line_length):
            global_idx = start_idx + i
            if i < n_aa_in_line:
                aa = seq_sub[i]
                in_pep = pep_positions[global_idx]
                # Draw colored or white box
                rect_fc = colors[in_pep - 1] if in_pep else '#f5f5f5'
                rect_ec = 'black'
                ax.add_patch(plt.Rectangle((i, 0), 1, 1,
                            facecolor=rect_fc if in_pep else '#ffffff',
                            edgecolor=rect_ec,
                            lw=1.5 if in_pep else 0.7,
                            alpha=0.7 if in_pep else 1.0,
                            zorder=2 if in_pep else 1,
                            linewidth=1.3))
                # Draw amino acid text centered in box
                ax.text(i + 0.5, 0.5, aa, ha='center', va='center', color='black',
                        fontsize=16, fontfamily='monospace', weight='bold', zorder=4)
            else:
                # Empty box for padding
                ax.add_patch(plt.Rectangle((i, 0), 1, 1,
                            facecolor='#f5f5f5',
                            edgecolor='black',
                            lw=0.6,
                            alpha=1.0,
                            zorder=1,
                            linewidth=1.0))

        ax.set_xlim(0, line_length)
        ax.set_ylim(0, 1.23)
        ax.set_yticks([])
        # Residue number ticks under boxes
        residue_ticks = list(range(0, line_length, 10))
        label_start = start_idx + 1  # 1-based
        xtick_labels = []
        for x in residue_ticks:
            if x < n_aa_in_line:
                xtick_labels.append(str(label_start + x))
            else:
                xtick_labels.append('')
        ax.set_xticks([x + 0.5 for x in residue_ticks])
        ax.set_xticklabels(xtick_labels, fontsize=12)
        for spine in ['top', 'right', 'left']:
            ax.spines[spine].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        # Remove the PrEST label/title from above (done via suptitle at top)

    # Build legend only
    legend_handles = []
    legend_labels = []
    for pep, idx in pep_indices.items():
        legend_handles.append(plt.Line2D([0], [0], color=colors[idx % len(colors)], lw=8))
        legend_labels.append(pep)
    if legend_handles:
        axes[-1, 0].legend(
            legend_handles,
            legend_labels,
            loc='upper center',
            bbox_to_anchor=(0.5, -0.30 + (0.3/fig_height)), # dynamically adjust by fig_height
            ncol=3,
            fontsize=11,
            frameon=False
        )
    plt.tight_layout(h_pad=0.8, rect=[0, 0, 1, 0.98])
    return fig

In [ ]:
fasta_file.head(20)

In [ ]:
# Plot map_peptide_sequence
map_peptide_sequence(abs_df, fasta_file, 'QR0330186_VWF|P04275')


In [ ]:
def plot_peptide_concentration_by_group(abs_df, sdrf_data_file, group_col, protein_name):
    """
    Plot peptide-level absolute protein concentration by an SDRF grouping column,
    with each peptide's boxplot on a separate (vertically stacked) subplot, 
    colored by the grouping variable.

    Parameters
    ----------
    abs_df : pandas.DataFrame
        Wide output from ``kdm.get_absolute_conc`` (MultiIndex + replicate columns).
    sdrf_data_file : pandas.DataFrame
        SDRF table with ``source name`` and *group_col*.
    group_col : str
        SDRF column to group/color by (e.g. ``factor value[disease]``).
    protein_name : str
        Filter to this ``Protein Name`` value.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Check that specified columns exist
    if group_col not in sdrf_data_file.columns:
        raise KeyError(f"Column '{group_col}' not found in sdrf_data_file.")
    if "source name" not in sdrf_data_file.columns:
        raise KeyError("Column 'source name' not found in sdrf_data_file.")

    # Convert abs_df to long format
    long_df = (
        abs_df.reset_index()
        .melt(
            id_vars=["qRePS", "Peptide Sequence", "Protein Name"],
            var_name="Replicate",
            value_name="Protein conc [pmol]",
        )
        .dropna(subset=["Protein conc [pmol]"])
    )

    # Check Replicate <-> 'source name' mapping
    replicates = set(long_df["Replicate"])
    sources = set(sdrf_data_file["source name"])
    unmapped = replicates - sources
    can_map = len(unmapped) == 0
    print(f"All Replicate values in abs_df mapped to 'source name' in sdrf_data_file: {can_map}")
    if unmapped:
        print(f"  Unmapped replicates ({len(unmapped)}): {sorted(unmapped)}")

    # Merge to add group_col metadata
    meta_cols = ["source name", group_col]
    meta_df = sdrf_data_file[meta_cols].drop_duplicates()
    plot_df = long_df.merge(
        meta_df,
        left_on="Replicate",
        right_on="source name",
        how="left",
    )

    # Filter for specified protein
    protein_df = plot_df[plot_df["Protein Name"] == protein_name].copy()
    if protein_df.empty:
        print(f"No data found for Protein Name: '{protein_name}'")
        print("Available Protein Names:", plot_df["Protein Name"].unique())
        return

    # Get unique peptide sequences
    peptides = protein_df["Peptide Sequence"].unique()
    n_peptides = len(peptides)
    if n_peptides == 0:
        print(f"No peptide data found for Protein Name: '{protein_name}'")
        return

    # Build a color palette for the groups
    groups_all = protein_df[group_col].dropna().unique()
    palette = sns.color_palette(n_colors=len(groups_all))
    group2color = dict(zip(groups_all, palette))

    # Set up vertical subplots, sharing x-axis
    fig, axes = plt.subplots(
        n_peptides, 1, 
        figsize=(10, 3 * n_peptides), 
        sharex=True
    )

    if n_peptides == 1:
        axes = [axes]  # Make it iterable even if single subplot

    for ax, pep in zip(axes, peptides):
        pep_df = protein_df[protein_df["Peptide Sequence"] == pep]
        # Order categories for consistency
        group_order = sorted(pep_df[group_col].dropna().unique(), key=lambda x: str(x))
        # Boxplot with hue to color by group_col
        sns.boxplot(
            data=pep_df,
            x=group_col,
            y="Protein conc [pmol]",
            ax=ax,
            order=group_order,
            palette=group2color,
        )

        # Add average concentration (mean) value in the middle of each box
        for idx, group in enumerate(group_order):
            group_data = pep_df[pep_df[group_col] == group]["Protein conc [pmol]"]
            if not group_data.empty:
                mean_val = group_data.mean()
                pos = idx
                ax.text(pos, mean_val, f"{mean_val:.2f}", 
                        ha='center', va='center', 
                        fontsize=9, fontweight="bold", color='black', 
                        bbox=dict(facecolor='white', edgecolor='none', pad=0.3, alpha=0.7)
                )

        # Append protein_name to the peptide title
        ax.set_title(f"{pep}|{protein_name}", fontsize=10, loc='left')
        ax.set_ylabel("Protein conc [pmol]")
        ax.set_xlabel(group_col if ax == axes[-1] else "")
        ax.tick_params(axis='x', rotation=30)
        # Remove legend for each subplot (not needed)
        if ax.get_legend() is not None:
            ax.legend_.remove()
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

In [ ]:
def plot_median_peptide_concentration_by_group(abs_df, sdrf_data_file, group_col, protein_name):
    """
    Plot the median peptide concentration by group for each peptide of a given protein as line plots,
    with group median values.

    Also plots error bars as the standard error of the mean (SEM) for each group.

    Args:
        abs_df (pd.DataFrame): Peptide concentration data.
        sdrf_data_file (pd.DataFrame): Sample metadata with mapping for 'Replicate' and group information.
        group_col (str): Name of the metadata column to group samples by (e.g., "factor value[disease]").
        protein_name (str): Name of the protein (in "Protein Name" column) to filter and plot.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd

    # Validate required columns exist
    if group_col not in sdrf_data_file.columns:
        raise KeyError(f"Column '{group_col}' not found in sdrf_data_file.")
    if "source name" not in sdrf_data_file.columns:
        raise KeyError("Column 'source name' not found in sdrf_data_file.")

    # Convert abs_df to long format for easier processing
    long_df = (
        abs_df.reset_index()
        .melt(
            id_vars=["qRePS", "Peptide Sequence", "Protein Name"],
            var_name="Replicate",
            value_name="Protein conc [pmol]",
        )
        .dropna(subset=["Protein conc [pmol]"])
    )

    # Map 'Replicate' in abs_df to 'source name' in metadata, warn if mapping is incomplete
    replicates = set(long_df["Replicate"])
    sources = set(sdrf_data_file["source name"])
    unmapped = replicates - sources
    print(
        f"All Replicate values mapped to 'source name': {len(unmapped)==0}"
    )
    if unmapped:
        print(f"  Unmapped replicates ({len(unmapped)}): {sorted(unmapped)}")

    # Merge metadata group info into long_df
    meta = sdrf_data_file[["source name", group_col]].drop_duplicates()
    plot_df = long_df.merge(
        meta,
        left_on="Replicate",
        right_on="source name",
        how="left",
    )

    # Filter to the specified protein
    protein_df = plot_df[plot_df["Protein Name"] == protein_name].copy()
    if protein_df.empty:
        print(f"No data found for Protein Name: '{protein_name}'")
        print("Available Protein Names:", plot_df["Protein Name"].unique())
        return

    # Gather peptides
    peptides = protein_df["Peptide Sequence"].unique()
    if len(peptides) == 0:
        print(f"No peptide data found for Protein Name: '{protein_name}'")
        return

    # Color for each peptide
    palette = sns.color_palette("tab10", len(peptides))
    peptide_colors = dict(zip(peptides, palette))

    # ----- Compute both median and SEM by peptide x group -----
    # .agg returns a DataFrame with columns for each agg function
    agg_stats = (
        protein_df
        .groupby(["Peptide Sequence", group_col])["Protein conc [pmol]"]
        .agg(['median', 'mean', 'std', 'count'])
        .reset_index()
    )
    # SEM = std / sqrt(count)
    agg_stats["sem"] = agg_stats["std"] / agg_stats["count"].pow(0.5)

    group_order = sorted(agg_stats[group_col].dropna().unique(), key=str)

    plt.figure(figsize=(12, 8))
    for pep in peptides:
        pep_stats = agg_stats[agg_stats["Peptide Sequence"] == pep].set_index(group_col).reindex(group_order)
        y_median_vals = pep_stats["median"].values
        # Use SEM for error bar; if NaN, use 0; align to group_order
        y_sem_vals = pep_stats["sem"].values if "sem" in pep_stats.columns else np.zeros_like(y_median_vals)
        # For x locations, use numpy arange to center errorbar at each group
        x_pos = np.arange(len(group_order))

        # Plot line through medians, with error bars for SEM
        plt.errorbar(
            group_order,
            y_median_vals,
            yerr=y_sem_vals,
            marker="o",
            label=pep,
            color=peptide_colors[pep],
            capsize=4,
            linestyle='-',
        )

        # Labels above each point have been removed as per instruction

    plt.title(f"Median peptide concentration by group for {protein_name}")
    plt.xlabel(group_col)
    plt.ylabel("Median Protein/Peptide conc [pmol]")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.legend(title="Peptide Sequence")
    plt.show()

In [ ]:
plot_median_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    group_col="factor value[disease]",
    protein_name="QR0290109_EGFR|P00533",  # from abs_df.reset_index()['Protein Name'].unique()
)


In [ ]:
def plot_all_median_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    group_col="factor value[disease]",
    pdf_path="median_peptide_concentration_by_group.pdf",
    verbose=False,
):
    """
    Plot each protein's median peptide concentration by group,
    saving all plots to a single multi-page PDF (A4 horizontal size per page).
    Does not display plots inline.
    """
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    import warnings

    # A4 in inches (width, height), landscape
    A4_WIDTH, A4_HEIGHT = 11.69, 8.27

    # Patch plt.show to prevent plots from displaying
    original_show = plt.show
    plt.show = lambda *a, **kw: None

    # Suppress FutureWarnings about palette/hue, as in similar functions
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.",
            category=FutureWarning,
            module="seaborn"
        )

        protein_names = abs_df.reset_index()["Protein Name"].unique()
        with PdfPages(pdf_path) as pdf:
            for pname in protein_names:
                if verbose:
                    print(f"Plotting {pname}")
                try:
                    plot_median_peptide_concentration_by_group(
                        abs_df,
                        sdrf_data_file,
                        group_col=group_col,
                        protein_name=pname,
                    )
                    fig = plt.gcf()
                    fig.set_size_inches(A4_WIDTH, A4_HEIGHT)
                    pdf.savefig(fig)
                    plt.close(fig)
                except Exception as e:
                    print(f"Error plotting {pname}: {e}")
    # Restore original plt.show (if needed elsewhere)
    plt.show = original_show

# Show example
plot_all_median_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    group_col="factor value[disease]",
    pdf_path="data/DA4000/median_peptide_concentration_by_group.pdf",
)


In [ ]:
plot_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    group_col="factor value[disease]",
    protein_name="QR0290109_EGFR|P00533",  # from abs_df.reset_index()['Protein Name'].unique()
)


In [ ]:
def plot_all_peptide_concentration_by_group(
    abs_df,
    sdrf_data_file,
    group_col="factor value[disease]",
    pdf_path="peptide_concentration_by_group.pdf",
    verbose=False,
):
    """
    Plot every protein and save all figures to one PDF via plot_peptide_concentration_by_group,
    only saving to file (do not show the plot).
    Suppresses deprecation warnings related to Seaborn's palette/hue logic.
    Ensures every PDF page (figure) is A4 size.
    """
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    import warnings

    # A4 in inches for matplotlib
    A4_WIDTH, A4_HEIGHT = 8.27, 11.69

    # Patch plt.show to do nothing during this function
    original_show = plt.show
    plt.show = lambda *a, **kw: None

    # Suppress specific FutureWarnings about palette/hue from seaborn boxplot
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.",
            category=FutureWarning,
            module="seaborn"
        )

        protein_names = abs_df.reset_index()["Protein Name"].unique()
        saved = 0
        skipped = []

        with PdfPages(pdf_path) as pdf:
            for i, pname in enumerate(protein_names):
                try:
                    # Create a new A4-sized figure before each plot
                    plt.figure(figsize=(A4_WIDTH, A4_HEIGHT))
                    plot_peptide_concentration_by_group(
                        abs_df,
                        sdrf_data_file,
                        group_col=group_col,
                        protein_name=pname,
                    )
                    fig = plt.gcf()
                    pdf.savefig(fig)
                    plt.close(fig)
                    saved += 1
                except ValueError:
                    skipped.append(pname)

    # Restore original plt.show
    plt.show = original_show

    print(f"Saved {saved} plots to {pdf_path}")
    if skipped:
        print(f"Skipped {len(skipped)} protein(s) with no data.")
    return pdf_path

plot_all_peptide_concentration_by_group(
    abs_df=abs_df,
    sdrf_data_file=sdrf_data_file,
    group_col="factor value[disease]",
    pdf_path="data/DA4000/peptide_concentration_by_group.pdf",
)